In [2]:
import torch

# 1. Setup device (Handles NVIDIA, Apple Silicon, or CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") # For Mac M1/M2/M3 chips
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# 2. Move a Tensor to the GPU
x = torch.randn(3, 3).to(device)

# 3. Move a Model to the GPU
model = torch.nn.Linear(3, 1).to(device)

# 4. Perform operation (Must be on the same device!)
output = model(x)
print(output.device) # Should be 'cuda:0' or 'mps:0'

Using device: mps
mps:0


In [1]:
import torch

# Check if Apple Silicon GPU is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU")

# Move a tensor to your Mac's GPU
x = torch.randn(5, 5).to(device)
print(x.device) # Should output 'mps:0'

Using MPS (Apple Silicon GPU)
mps:0


In [4]:
import torch
import time

# Create two large matrices (10,000 x 10,000)
size = 10000
cpu_a = torch.randn(size, size)
cpu_b = torch.randn(size, size)

# --- CPU TEST ---
start_cpu = time.time()
cpu_result = torch.matmul(cpu_a, cpu_b)
end_cpu = time.time()
print(f"CPU Time: {end_cpu - start_cpu:.4f} seconds")

# --- MPS TEST (GPU) ---
device = torch.device("mps")
mps_a = cpu_a.to(device)
mps_b = cpu_b.to(device)

# Warm-up (The first GPU call always has a slight overhead)
_ = torch.matmul(mps_a, mps_b)

start_mps = time.time()
mps_result = torch.matmul(mps_a, mps_b)
# Synchronize is important for accurate timing on MPS
torch.mps.synchronize() 
end_mps = time.time()

print(f"MPS Time: {end_mps - start_mps:.4f} seconds")

CPU Time: 1.1781 seconds
MPS Time: 1.8391 seconds


In [6]:
import torch
import time
import statistics

size = 10000
repeats = 5

# Check MPS
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is not available on this system")

# Create data
cpu_a = torch.randn(size, size, dtype=torch.float32)
cpu_b = torch.randn(size, size, dtype=torch.float32)

# --- CPU warm-up ---
_ = torch.matmul(cpu_a, cpu_b)

cpu_times = []
for _ in range(repeats):
    start = time.perf_counter()
    _ = torch.matmul(cpu_a, cpu_b)
    end = time.perf_counter()
    cpu_times.append(end - start)

# --- MPS setup ---
device = torch.device("mps")
mps_a = cpu_a.to(device)
mps_b = cpu_b.to(device)

# --- MPS warm-up ---
_ = torch.matmul(mps_a, mps_b)
torch.mps.synchronize()   # important

mps_times = []
for _ in range(repeats):
    start = time.perf_counter()
    _ = torch.matmul(mps_a, mps_b)
    torch.mps.synchronize()   # important
    end = time.perf_counter()
    mps_times.append(end - start)

print("CPU times:", cpu_times)
print("CPU median:", statistics.median(cpu_times))

print("MPS times:", mps_times)
print("MPS median:", statistics.median(mps_times))

CPU times: [1.1535881659999632, 1.185143417000063, 1.1676702920000253, 1.1314464580000276, 1.1717002919999686]
CPU median: 1.1676702920000253
MPS times: [0.905752833000065, 0.9069571669999732, 0.9053079169999592, 0.9098223750000898, 0.9123838339999111]
MPS median: 0.9069571669999732
